In [1]:
# Saving regional mortality for all ensemble members

In [2]:
import os
import xarray as xr
import regionmask
import matplotlib.pyplot as plt
import numpy as np

In [3]:
# === Latitude weighting sum ===
def weighted_sum(da):
    weights = np.cos(np.deg2rad(da.lat))
    weights.name = "weights"
    new_da = da.weighted(weights).sum(("lon", "lat"))
    return new_da

In [42]:
# === Path config ===
MORTALITY_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

BMR_DIR = "/glade/work/awells/air_quality/BMR/"
region_mask = xr.open_dataarray(f"{BMR_DIR}GBD_Region_Masks_0.10_popgrid.nc")
country_mask = xr.open_dataarray(f"{BMR_DIR}GBD_Country_Masks_0.10_popgrid_newlabels.nc")

# === Main loop ===
for scenario in SCENARIOS:
    ensembles = []
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2069"
        elif scenario == "SSP245":
            dates = "2020-2069"

        da = xr.open_dataarray(f"{MORTALITY_DIR}Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

        regions = []
        # Loop over regions and sum the mortality for each region
        for i in range(len(region_mask["region"])):
            mask = region_mask.isel(region=i)
            M_region = weighted_sum(xr.where(region_mask.isel(region=i) == 1, da.sel(quantile="median"), np.nan)) # mortality of country
            regions.append(M_region.drop_vars("region", errors='ignore'))
        mortality_region = xr.concat(regions, dim=xr.DataArray(region_mask["region"], dims="region", name="region"))

        ensembles.append(mortality_region)

    regions_ens = xr.concat(ensembles, dim=xr.DataArray(np.arange(1, len(ensembles)+1), dims="ensemble", name="ensemble"))
    print(f"Saving regional mortality to {MORTALITY_DIR}")
    regions_ens.to_netcdf(f"{MORTALITY_DIR}Mortality_Regional_sum_CESM2_{scenario}_{dates}.nc")

Processing ARISE, Ensemble 01
Processing ARISE, Ensemble 02
Processing ARISE, Ensemble 03
Processing ARISE, Ensemble 04
Processing ARISE, Ensemble 05
Processing ARISE, Ensemble 06
Processing ARISE, Ensemble 07
Processing ARISE, Ensemble 08
Processing ARISE, Ensemble 09
Processing ARISE, Ensemble 10
Saving regional mortality to /glade/work/awells/air_quality/CESM/mortality/
Processing SSP245, Ensemble 01
Processing SSP245, Ensemble 02
Processing SSP245, Ensemble 03
Processing SSP245, Ensemble 04
Processing SSP245, Ensemble 05
Processing SSP245, Ensemble 06
Processing SSP245, Ensemble 07
Processing SSP245, Ensemble 08
Processing SSP245, Ensemble 09
Processing SSP245, Ensemble 10
Saving regional mortality to /glade/work/awells/air_quality/CESM/mortality/
